# Inspect Words MVP Database

这个 notebook 用于查看当前 SQLite 数据库内容，重点检查义项级状态是否正确更新。

In [4]:
from pathlib import Path
import sqlite3
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DB_PATH = PROJECT_ROOT / "data" / "words_mvp_v2.sqlite3"
DB_PATH

PosixPath('/home/pillar/projects_linux/words-mvp/data/words_mvp_v2.sqlite3')

In [5]:
assert DB_PATH.exists(), f"Database not found: {DB_PATH}"
con = sqlite3.connect(DB_PATH)
con.row_factory = sqlite3.Row

def q(sql, params=None):
    return pd.read_sql_query(sql, con, params=params or {})

## 1. 查看当前有哪些表

In [6]:
q("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name
""")

,name
0,documents
1,lexemes
2,sqlite_sequence
3,text_occurrences
4,user_sense_events
5,user_sense_states
6,word_senses


## 2. 查看每张核心表的数据量

In [7]:
tables = [
    "documents",
    "lexemes",
    "word_senses",
    "text_occurrences",
    "user_sense_states",
    "user_sense_events",
]

rows = []
for table in tables:
    count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    rows.append({"table": table, "row_count": count})
pd.DataFrame(rows)

,table,row_count
0,documents,1
1,lexemes,30
2,word_senses,30
3,text_occurrences,30
4,user_sense_states,1
5,user_sense_events,1


## 3. 最近导入的文档

In [8]:
q("""
SELECT id, filename, LENGTH(content) AS content_length, created_at
FROM documents
ORDER BY id DESC
LIMIT 20
""")

,id,filename,content_length,created_at
0,1,test_text.txt,3609,2026-05-17 07:11:26


## 4. 词条与词条常见程度

In [9]:
q("""
SELECT id, lemma, pos, frequency_rank, frequency_score, frequency_source, created_at
FROM lexemes
ORDER BY frequency_rank IS NULL, frequency_rank ASC, lemma ASC
LIMIT 50
""")

,id,lemma,pos,frequency_rank,frequency_score,frequency_source,created_at
0,6,station,n,800.0,4.0,ecdict.frq,2026-05-17 07:11:26
1,26,flight,n,1300.0,4.0,ecdict.frq,2026-05-17 07:11:26
2,8,equipment,n,1600.0,4.0,ecdict.frq,2026-05-17 07:11:26
3,15,technology,n,1800.0,4.0,ecdict.frq,2026-05-17 07:11:26
4,14,successful,adj,1900.0,4.0,ecdict.frq,2026-05-17 07:11:26
5,29,critical,adj,2200.0,4.0,ecdict.frq,2026-05-17 07:11:26
6,28,mission,n,2300.0,4.0,ecdict.frq,2026-05-17 07:11:26
7,13,scientific,adj,2600.0,3.0,ecdict.frq,2026-05-17 07:11:26
8,18,launch,n/v,3200.0,4.0,ecdict.frq,2026-05-17 07:11:26
9,7,exploration,n,6500.0,2.0,ecdict.frq,2026-05-17 07:11:26


## 5. 词条对应的具体义项

In [10]:
q("""
SELECT
    ws.id AS sense_id,
    l.lemma,
    ws.meaning_zh,
    ws.definition_en,
    ws.pos,
    ws.source,
    ws.source_sense_id,
    ws.sense_rank,
    ws.sense_key
FROM word_senses ws
JOIN lexemes l ON l.id = ws.lexeme_id
ORDER BY l.lemma ASC, ws.sense_rank ASC
LIMIT 100
""")

,sense_id,lemma,meaning_zh,definition_en,pos,source,source_sense_id,sense_rank,sense_key
0,27,booster,助推器,a rocket that gives extra power for launch,n,ecdict,ecdict:booster:1,1,ecdict:booster:9694431efb85
1,24,cargo,货物,goods carried by ship aircraft or vehicle,n,ecdict,ecdict:cargo:1,1,ecdict:cargo:f961c1d6fe12
2,19,component,ECDICT 未覆盖该词，需补充词典数据,,,missing_ecdict,missing:component,999999,missing:component
3,29,critical,关键的,very important; expressing criticism,adj,ecdict,ecdict:critical:1,1,ecdict:critical:620691f3865f
4,25,deliver,ECDICT 未覆盖该词，需补充词典数据,,,missing_ecdict,missing:deliver,999999,missing:deliver
5,1,demonstration,ECDICT 未覆盖该词，需补充词典数据,,,missing_ecdict,missing:demonstration,999999,missing:demonstration
6,30,endeavor,努力,a serious attempt or effort,n/v,ecdict,ecdict:endeavor:1,1,ecdict:endeavor:22bffc183a04
7,8,equipment,设备,the things needed for a particular purpose,n,ecdict,ecdict:equipment:1,1,ecdict:equipment:ef0eab7b97ed
8,20,essential,ECDICT 未覆盖该词，需补充词典数据,,,missing_ecdict,missing:essential,999999,missing:essential
9,4,expedition,远征,a journey organized for a particular purpose; ...,n,ecdict,ecdict:expedition:1,1,ecdict:expedition:968056ab0d38


## 6. 文本中实际出现的位置

In [11]:
q("""
SELECT
    o.id AS occurrence_id,
    o.document_id,
    l.lemma,
    o.surface,
    o.sentence_index,
    o.sentence,
    o.created_at
FROM text_occurrences o
LEFT JOIN lexemes l ON l.id = o.lexeme_id
ORDER BY o.id DESC
LIMIT 50
""")

,occurrence_id,document_id,lemma,surface,sentence_index,sentence,created_at
0,30,1,endeavor,endeavor,13,"""The International Space Station is a truly gl...",2026-05-17 07:11:26
1,29,1,critical,critical,14,It serves both as a proving ground for scienti...,2026-05-17 07:11:26
2,28,1,mission,mission,7,"""This mission includes everything from water p...",2026-05-17 07:11:26
3,27,1,booster,booster,1,Running three days late because of bad weather...,2026-05-17 07:11:26
4,26,1,flight,flight,2,"Two and a half minutes after liftoff, the rock...",2026-05-17 07:11:26
5,25,1,deliver,delivering,10,The Cargo Dragon is delivering more than 3 ton...,2026-05-17 07:11:26
6,24,1,cargo,cargo,0,SpaceX launched an unpiloted Dragon cargo ship...,2026-05-17 07:11:26
7,23,1,touchdown,touchdown,2,"Two and a half minutes after liftoff, the rock...",2026-05-17 07:11:26
8,22,1,spacewalk,spacewalk,10,The Cargo Dragon is delivering more than 3 ton...,2026-05-17 07:11:26
9,21,1,represent,represents,12,"""And that represents the work of over 5,000 re...",2026-05-17 07:11:26


## 7. 用户对具体义项的当前状态

In [12]:
q("""
SELECT
    uss.user_id,
    uss.sense_id,
    l.lemma,
    ws.meaning_zh,
    uss.status,
    uss.mastery_level,
    uss.source_document_id,
    uss.source_occurrence_id,
    uss.last_seen_at,
    uss.last_action_at,
    uss.updated_at
FROM user_sense_states uss
JOIN word_senses ws ON ws.id = uss.sense_id
JOIN lexemes l ON l.id = ws.lexeme_id
ORDER BY uss.updated_at DESC, uss.id DESC
""")

,user_id,sense_id,lemma,meaning_zh,status,mastery_level,source_document_id,source_occurrence_id,last_seen_at,last_action_at,updated_at
0,default,1,demonstration,ECDICT 未覆盖该词，需补充词典数据,learning,1,1,1,2026-05-17 07:14:38,2026-05-17 07:14:38,2026-05-17 07:14:38


## 8. 用户操作历史

In [13]:
q("""
SELECT
    e.id,
    e.user_id,
    e.sense_id,
    l.lemma,
    ws.meaning_zh,
    e.event_type,
    e.from_status,
    e.to_status,
    e.document_id,
    e.occurrence_id,
    e.created_at
FROM user_sense_events e
JOIN word_senses ws ON ws.id = e.sense_id
JOIN lexemes l ON l.id = ws.lexeme_id
ORDER BY e.id DESC
LIMIT 100
""")

,id,user_id,sense_id,lemma,meaning_zh,event_type,from_status,to_status,document_id,occurrence_id,created_at
0,1,default,1,demonstration,ECDICT 未覆盖该词，需补充词典数据,add_to_book,None,learning,1,1,2026-05-17 07:14:38


## 9. 检查某个词的所有义项和用户状态

修改 `target_lemma` 后重新运行下面的 cell。

In [14]:
target_lemma = "station"

q("""
SELECT
    l.lemma,
    ws.id AS sense_id,
    ws.meaning_zh,
    ws.definition_en,
    ws.sense_rank,
    COALESCE(uss.status, 'no_user_state') AS user_status,
    uss.mastery_level,
    uss.updated_at
FROM lexemes l
JOIN word_senses ws ON ws.lexeme_id = l.id
LEFT JOIN user_sense_states uss ON uss.sense_id = ws.id AND uss.user_id = 'default'
WHERE l.lemma = :lemma
ORDER BY ws.sense_rank ASC
""", {"lemma": target_lemma})

,lemma,sense_id,meaning_zh,definition_en,sense_rank,user_status,mastery_level,updated_at
0,station,6,站,a place or building where a service is based; ...,1,no_user_state,None,None


## 10. 检查某个 occurrence 的上下文

In [15]:
target_occurrence_id = 1

q("""
SELECT
    o.id AS occurrence_id,
    l.lemma,
    o.surface,
    o.sentence,
    o.context
FROM text_occurrences o
LEFT JOIN lexemes l ON l.id = o.lexeme_id
WHERE o.id = :occurrence_id
""", {"occurrence_id": target_occurrence_id})

,occurrence_id,lemma,surface,sentence,context
0,1,demonstration,demonstrations,"""The ISS has enabled more than 4,000 different...",The Cargo Dragon is delivering more than 3 ton...


## 11. 关闭数据库连接

In [16]:
con.close()